In [9]:
!pip install roboflow

In [10]:
!pip install ultralytics

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import os
import json
import time
import random
import numpy as np
import shutil
from datetime import datetime
from ultralytics import YOLO
from roboflow import Roboflow
import glob

In [ ]:
import yaml
import os
import glob
import shutil
from pathlib import Path

EXPERIMENTS_DIR = "/kaggle/working/runs/detect"
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)

# ==========================================
# Option 1: Load Dataset via Roboflow (Optional)
# ==========================================
# Uncomment and fill below if downloading directly from Roboflow:
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
# project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
# dataset = project.version(1).download("yolov8")

# ==========================================
# Option 2: Automatic Detection & Setup for Loaded Dataset
# ==========================================
# Search for any existing data.yaml in /kaggle/working/ or /kaggle/input/
existing_yamls = glob.glob('/kaggle/working/**/data.yaml', recursive=True) +                  glob.glob('/kaggle/input/**/data.yaml', recursive=True)
existing_yamls = [y for y in existing_yamls if os.path.abspath(y) != '/kaggle/working/data.yaml']

if existing_yamls:
    found_yaml = existing_yamls[0]
    print(f"✅ Found dataset configuration at: {found_yaml}")
    with open(found_yaml, 'r') as f:
        yaml_data = yaml.safe_load(f)
    yaml_dir = os.path.dirname(found_yaml)
    for split in ['train', 'val', 'test']:
        if split in yaml_data and yaml_data[split]:
            split_path = yaml_data[split]
            if not os.path.isabs(split_path):
                split_path = os.path.normpath(os.path.join(yaml_dir, split_path))
            yaml_data[split] = split_path
else:
    print("ℹ️ No data.yaml found automatically. Using configured dataset paths...")
    yaml_data = {
        'train': '/kaggle/input/datasets/sujityp/rdd2022/datasets/train',
        'val': '/kaggle/input/datasets/sujityp/rdd2022/datasets/valid',
        'test': '/kaggle/input/datasets/sujityp/rdd2022/datasets/test',
        'nc': 5,
        'names': ['longitudinal crack', 'transverse crack', 'alligator crack', 'other corruption', 'pothole']
    }

# ==========================================
# Fix Read-Only Filesystem Cache Error for Kaggle (/kaggle/input)
# ==========================================
# Ultralytics YOLO attempts to write labels.cache directly in the dataset folder.
# If the dataset is located inside /kaggle/input (read-only), YOLO will crash with FileNotFoundError/PermissionError.
# To prevent this, we mirror read-only dataset splits to /kaggle/working/dataset_writable/.
WRITABLE_DATASET_ROOT = "/kaggle/working/dataset_writable"

for split_key in ['train', 'val', 'test']:
    if split_key not in yaml_data or not yaml_data[split_key]:
        continue
    orig_path = yaml_data[split_key]
    if orig_path.startswith('/kaggle/input'):
        split_name = os.path.basename(orig_path.rstrip('/'))
        dest_split_path = os.path.join(WRITABLE_DATASET_ROOT, split_name)
        dest_images = os.path.join(dest_split_path, 'images')
        dest_labels = os.path.join(dest_split_path, 'labels')
        os.makedirs(dest_images, exist_ok=True)
        os.makedirs(dest_labels, exist_ok=True)
        
        orig_images = os.path.join(orig_path, 'images') if os.path.exists(os.path.join(orig_path, 'images')) else orig_path
        orig_labels = os.path.join(orig_path, 'labels') if os.path.exists(os.path.join(orig_path, 'labels')) else orig_path.replace('images', 'labels')
        
        if os.path.exists(orig_images) and not os.listdir(dest_images):
            print(f"🔄 Mirroring {split_key} images from {orig_images} -> {dest_images}")
            for img_file in glob.glob(os.path.join(orig_images, '*.*')):
                dest_file = os.path.join(dest_images, os.path.basename(img_file))
                if not os.path.exists(dest_file):
                    try:
                        os.symlink(img_file, dest_file)
                    except (OSError, AttributeError):
                        shutil.copy2(img_file, dest_file)
                        
        if os.path.exists(orig_labels) and not os.listdir(dest_labels):
            print(f"🔄 Copying {split_key} labels from {orig_labels} -> {dest_labels}")
            for lbl_file in glob.glob(os.path.join(orig_labels, '*.txt')):
                shutil.copy2(lbl_file, os.path.join(dest_labels, os.path.basename(lbl_file)))
                
        yaml_data[split_key] = dest_split_path

DATA_YAML = '/kaggle/working/data.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print("
🚀 Final Dataset Configuration (data.yaml):")
with open(DATA_YAML, 'r') as f:
    print(f.read())

val_images = glob.glob(os.path.join(yaml_data.get('val', ''), 'images', '*.jpg')) +              glob.glob(os.path.join(yaml_data.get('val', ''), '*.jpg')) +              glob.glob('/kaggle/working/**/images/*.jpg', recursive=True) +              glob.glob('/kaggle/input/**/images/*.jpg', recursive=True)
DUMMY_IMAGE_PATH = val_images[0] if val_images else None
print(f"🎯 Dummy image for warmup set to: {DUMMY_IMAGE_PATH}")

In [13]:
MODELS = ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]
EPOCHS = 15         
BATCH_SIZE = 64      # 
IMAGE_SIZE = 640
LEARNING_RATE = 0.01
OPTIMIZER = "auto"  
SEED = 42



In [14]:
def train_one_model(weights_name: str) -> dict:
    model_name = weights_name.replace(".pt", "")
    run_name = f"{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
 
    print(f"\n{'='*40}")
    print(f"🚀 Start training {model_name} 🚀")
    print(f"{'='*40}")
 
    # أ. التدريب
    model = YOLO(weights_name)
    model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMAGE_SIZE,
        lr0=LEARNING_RATE,
        optimizer=OPTIMIZER,
        seed=SEED,
        project=EXPERIMENTS_DIR,
        name=run_name,
        device=0, # تفعيل كارت الشاشة
        exist_ok=True
    )
 
    # ب. استخراج أفضل نموذج
    best_weights_path = os.path.join(EXPERIMENTS_DIR, run_name, "weights", "best.pt")
    best_model = YOLO(best_weights_path)
    
    # ج. التقييم على بيانات التحقق (Validation)
    metrics = best_model.val(data=DATA_YAML)
    map50 = metrics.box.map50
    map50_95 = metrics.box.map
    precision = metrics.box.mp
    recall = metrics.box.mr
 
    # د. حساب حجم النموذج بالميجابايت
    model_size_mb = os.path.getsize(best_weights_path) / (1024 * 1024)
    
    # هـ. حساب سرعة الاستجابة (FPS & Inference Time)
    fps = None
    avg_inference_ms = None
    
    if DUMMY_IMAGE_PATH and os.path.exists(DUMMY_IMAGE_PATH):
        # Warm-up run (لتجهيز كارت الشاشة)
        best_model.predict(DUMMY_IMAGE_PATH, verbose=False)
        
        start = time.time()
        n_runs = 20
        for _ in range(n_runs):
            best_model.predict(DUMMY_IMAGE_PATH, verbose=False)
        elapsed = time.time() - start
        
        avg_inference_ms = (elapsed / n_runs) * 1000
        fps = 1000 / avg_inference_ms
 
    # و. تجميع النتائج
    result = {
        "Model": model_name,
        "mAP50": round(map50, 4),
        "mAP50-95": round(map50_95, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "Size_MB": round(model_size_mb, 2),
        "Inference_ms": round(avg_inference_ms, 2) if avg_inference_ms else None,
        "FPS": round(fps, 2) if fps else None,
        "Weights_Path": best_weights_path
    }
 
    print(f"\n📊 Results for {model_name}:")
    print(f"mAP50: {result['mAP50']} | mAP50-95: {result['mAP50-95']}")
    print(f"Precision: {result['Precision']} | Recall: {result['Recall']}")
    print(f"Size: {result['Size_MB']} MB | FPS: {result['FPS']}")
    
    return result

In [ ]:
all_results = []

for weights in MODELS:
    try:
        res = train_one_model(weights)
        all_results.append(res)
    except Exception as e:
        print(f"❌ Error during training {weights}: {e}")

df_results = pd.DataFrame(all_results)
df_results.to_csv('/kaggle/working/Final_Models_Comparison.csv', index=False)
print(df_results.to_string(index=False))

In [ ]:
if not df_results.empty:
    
    best_model_row = df_results.loc[df_results['mAP50'].idxmax()]
    best_weights_source = best_model_row['Weights_Path']
    
    
    best_weights_dest = '/kaggle/working/best_overall_model.pt'
    shutil.copy(best_weights_source, best_weights_dest)
    
    print(f"\n best model: {best_model_row['Model']} acc {best_model_row['mAP50']}")
  
    
    
    
    shutil.make_archive('/kaggle/working/RoadDamage_Project_Files', 'zip', '/kaggle/working/')
    